## Check Dataset Duration

In [ ]:
!pip install requests tqdm pandas

In [ ]:
import os
import zipfile
import requests
import pandas as pd
from tqdm import tqdm
from glob import glob

# CONFIGURATION

SYMBOL = "BTCUSDT"

INTERVALS = [
    "1m",
    "5m",
    "15m",
    "30m",
    "1h"
]

START_YEAR = 2024
END_YEAR = 2025

BASE_URL = "https://data.binance.vision/data/spot/monthly/klines"

GOOGLE_DRIVE_FOLDER = "/content/drive/MyDrive/Binance_Final_Datasets"

os.makedirs(GOOGLE_DRIVE_FOLDER, exist_ok=True)

# DOWNLOAD FUNCTION

def download(url, filename):

    if os.path.exists(filename):
        return

    response = requests.get(url, stream=True)

    if response.status_code != 200:
        return

    total = int(response.headers.get("content-length", 0))

    with open(filename, "wb") as f, tqdm(
        total=total,
        desc=os.path.basename(filename),
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:

        for chunk in response.iter_content(8192):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

# MAIN LOOP

for interval in INTERVALS:

    print(f"\nProcessing {interval}")

    temp_folder = f"/content/temp_{interval}"

    os.makedirs(temp_folder, exist_ok=True)

    merged = []

    for year in range(START_YEAR, END_YEAR + 1):

        months = range(1,13)

        if year == 2026:
            months = range(1,7)

        for month in months:

            filename = f"{SYMBOL}-{interval}-{year}-{month:02d}.zip"

            url = f"{BASE_URL}/{SYMBOL}/{interval}/{filename}"

            zip_path = os.path.join(temp_folder, filename)

            download(url, zip_path)

            if not os.path.exists(zip_path):
                continue

            with zipfile.ZipFile(zip_path) as z:

                csv_name = z.namelist()[0]

                df = pd.read_csv(
                    z.open(csv_name),
                    header=None
                )

                merged.append(df)

    final = pd.concat(
        merged,
        ignore_index=True
    )
    output = os.path.join(
        GOOGLE_DRIVE_FOLDER,
       f"{SYMBOL}_{interval}_2020_2026.csv"
    )
    final.to_csv(
        output,
        index=False,
      header=False
    )
    print(f"\nSaved : {output}")

print("\nALL DATASETS CREATED SUCCESSFULLY")

In [ ]:
from glob import glob
import os

BASE_DIR = "/content/drive/MyDrive/Binance_Data"

intervals = ["1m","5m","15m","30m","1h"]

for interval in intervals:

    files = sorted(glob(os.path.join(BASE_DIR, interval, "*.zip")))

    print("="*60)
    print(interval)
    print("Total ZIP Files :", len(files))

    if files:
        print("First :", os.path.basename(files[0]))
        print("Last  :", os.path.basename(files[-1]))